# HuggingFace Transformers: API, Models and Fine-Tuning Techniques

## Day 3

In [ ]:
import sys
import torch
import transformers
import PIL

print("Python version:", sys.version)
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("PIL version:", PIL.__version__)

##### Image Segmentation using Vision Transformers (ViT)

In [ ]:
from PIL import Image
import requests

print("Fetching image...")
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/segmentation_input.jpg"
image = Image.open(requests.get(url, stream=True).raw)
print("Image fetched successfully.")
image

In [ ]:
print(type(image))

In [ ]:
from transformers import pipeline
semantic_segmentation = pipeline("image-segmentation", 
                                 "nvidia/segformer-b1-finetuned-cityscapes-1024-1024",
                                 use_fast=True)

results = semantic_segmentation(image)
results

In [ ]:
results[2]["label"]

In [ ]:
results[2]["mask"]

In [ ]:
Image.composite(image, results[2]["mask"], results[2]["mask"])

--- 

##### Image Classification using Vision Transformers (ViT)

In [ ]:
!pip install matplotlib

In [ ]:
from transformers import pipeline
from PIL import Image
import requests

import matplotlib.pyplot as plt

# Load image classification pipeline
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    use_fast=True
)

# Load a sample image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
response = requests.get(url, stream=True).raw
image = Image.open(response)

image

In [ ]:

# Classify
results = classifier(image)
for result in results[:5]:
    print(f"{result['label']}: {result['score']:.3f}")


In [ ]:
image = Image.open("../../tulip2.jpeg")
image

In [ ]:
results = classifier(image)
for result in results[:5]:
    print(f"{result['label']}: {result['score']:.3f}")

In [ ]:

print("---")
# Try with your own images
image = Image.open("../../tulip2.jpeg")
results = classifier(image)
for result in results[:5]:
    print(f"{result['label']}: {result['score']:.3f}")

---

#### Image Enhancement

In [ ]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/cat.jpg"
image = Image.open(requests.get(url, stream=True).raw)

print(image.size)
image

In [ ]:
from transformers import pipeline
pipe = pipeline(task="image-to-image", model="caidas/swin2SR-lightweight-x2-64")

upscaled = pipe(image)
print(upscaled.size)
upscaled

In [ ]:
upscaled = pipe(upscaled)
print(upscaled.size)
upscaled

---

### Text-to-Text Generation

In [ ]:
from transformers import pipeline

grammar_pipeline = pipeline(
    "text2text-generation",
    model="t5-base")

incorrect_text = "He have been working here since five years."
input_text = f"correct grammar: {incorrect_text}"
result = grammar_pipeline(input_text, num_beams=5)

print("Original:", incorrect_text)
print("Corrected:", result[0]['generated_text'])

In [ ]:
!pip install protobuf

In [ ]:
from transformers import pipeline

# 1. Initialize the text2text-generation pipeline with a dedicated grammar model
grammar_pipeline = pipeline(
    "text2text-generation", 
    model="AventIQ-AI/t5-small-grammar-correction"
)

# Other models you can try:
# - AventIQ-AI/t5-base-grammar-correction
# - vennify/t5-base-grammar-correction
# - pszemraj/flan-t5-large-grammar-synthesis
# - Sajid030/t5-base-grammar-synthesis

# 2. Define the grammatically incorrect text
# Note: T5 models usually expect a task prefix like "correct grammar: "
incorrect_text = "She dont like to eat vegetables but she like fruits."
input_text = f"correct grammar: {incorrect_text}"

# 3. Generate the correction
result = grammar_pipeline(input_text, num_beams=5)

# 4. Print the corrected output
print("Original:", incorrect_text)
print("Corrected:", result[0]['generated_text'])


---

#### Working with ```datasets```

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("food101")
dataset

In [ ]:
dataset.keys() # Identify the splits in the dataset

In [ ]:
v = dataset["validation"]

In [ ]:
v.column_names

In [ ]:
v.features

In [ ]:
v.num_rows

In [ ]:
v.features["label"].num_classes  # Fetch the number of classes in the label feature

In [ ]:
train_dataset = dataset["train"]
train_dataset

In [ ]:
train_dataset[0]

In [ ]:
train_dataset.features["label"].names

In [ ]:
from random import randint

idx = randint(0, len(train_dataset) - 1)

print("Idx:", idx, ", Label:", train_dataset.features["label"].int2str(train_dataset[idx]["label"]))
train_dataset[idx]["image"]

In [ ]:
train_dataset.features["label"].int2str(train_dataset[0]["label"])

In [ ]:
dataset2 = load_dataset("ariG23498/flickr8k")
dataset2

In [ ]:
dataset2["train"].features

In [ ]:
from datasets import Dataset

d = Dataset.from_csv("reviews_simple.csv")
d

In [ ]:
d = load_dataset("csv", data_files="reviews_simple.csv")
d

---
## Model Fine-tuning examples

#### A Skeletal example

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    pipeline
)

# ==========================================
# 1. Prepare Your Custom Dataset
# ==========================================
# Sample dictionary (you can also load a CSV via pandas: pd.read_csv("data.csv"))
raw_data = {
    "text": [
        "I absolutely love this product! Highly recommended.",
        "Worst experience ever, totally disappointed.",
        "It's okay, nothing extraordinary.",
        "Great quality and fast delivery!",
        "Poor customer service and cheap quality.",
        "Fantastic service, will definitely buy again!"
    ],
    "label": [1, 0, 0, 1, 0, 1]  # 1: Positive, 0: Negative
}

# Convert dict to Hugging Face Dataset
#df = pd.DataFrame(raw_data)
dataset = Dataset.from_dict(raw_data)
dataset

In [ ]:
# Split into train and test/validation sets
dataset_split = dataset.train_test_split(test_size=0.33, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print(train_dataset)
print(eval_dataset)

In [ ]:

# ==========================================
# 2. Load Tokenizer & Model
# ==========================================
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Define label mappings
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2, 
    id2label=id2label, 
    label2id=label2id
)

# ==========================================
# 3. Tokenize Data
# ==========================================
def preprocess_function(examples):
    # Truncate and pad inputs to fit model context
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

print(tokenized_train)
print(tokenized_eval)

In [ ]:
from transformers import Trainer, TrainingArguments
# ==========================================
# 4. Set Training Arguments & Trainer
# ==========================================
training_args = TrainingArguments(
    output_dir="./custom_classification_model",
    eval_strategy="epoch",       # Evaluate at the end of every epoch
    save_strategy="epoch",       # Save checkpoint at the end of every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer
)


In [ ]:

# ==========================================
# 5. Fine-Tune & Save Model
# ==========================================
print("Starting training...")
trainer.train()


In [ ]:

# Save final model and tokenizer locally
trainer.save_model("./final_saved_model")
tokenizer.save_pretrained("./final_saved_model")
print("Model saved successfully!")

# ==========================================
# 6. Test with Inference Pipeline
# ==========================================
classifier = pipeline("text-classification", model="./final_saved_model")

test_text = "The quality was better than expected!"
prediction = classifier(test_text)

print(f"\nText: {test_text}")
print(f"Prediction: {prediction}")

---

#### Finetuning ModernBERT with custom dataset

In [ ]:
from datasets import load_dataset

# Dataset id from huggingface.co/dataset
dataset_id = "cornell-movie-review-data/rotten_tomatoes"

# Load raw dataset
raw_dataset = load_dataset(dataset_id)

print(f"Train dataset size: {len(raw_dataset['train'])}")
print(f"Test dataset size: {len(raw_dataset['test'])}")

In [ ]:
raw_dataset["train"].features

In [ ]:
from random import randrange

random_id = randrange(len(raw_dataset['train']))
raw_dataset['train'][random_id]


In [ ]:
from transformers import AutoTokenizer

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.model_max_length = 512 # set model_max_length to 512 as prompts are not longer than 1024 tokens

# Tokenize helper function
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, return_tensors="pt")

# Tokenize dataset
raw_dataset =  raw_dataset.rename_column("label", "labels") # to match Trainer
tokenized_dataset = raw_dataset.map(tokenize, batched=True,remove_columns=["text"])

print(tokenized_dataset["train"].features.keys())
# dict_keys(['input_ids', 'attention_mask','labels'])

In [ ]:
from transformers import AutoModelForSequenceClassification

# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Prepare model labels - useful for inference
labels = tokenized_dataset["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

In [ ]:
!pip install scikit-learn

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}

In [ ]:
from transformers import Trainer, TrainingArguments

# Define training args
training_args = TrainingArguments(
    output_dir= "modernbert-rotten-tomatoes",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
		num_train_epochs=5,
    bf16=True, # bfloat16 training 
    optim="adamw_torch_fused", # improved optimizer 
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # push to hub parameters
    #report_to="tensorboard",
    push_to_hub=False,

)

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

In [ ]:
# Start training
trainer.train()

In [ ]:
# Save processor and create model card
tokenizer.save_pretrained("modernbert-rotten-tomatoes")
trainer.create_model_card()
trainer.push_to_hub()

---

### Finetuning an Image Classifier (ViT) with custom Image dataset

#### The `Trainer` API

`Trainer` is a complete training and evaluation loop for Transformers’ PyTorch models. Plug a model, preprocessor, dataset, and training arguments into `Trainer` and let it handle the rest to start training faster.

The Trainer simplifies the overhead of implementing our training loop for our training workflow using PyTorch models.

Trainer contains all the necessary components of a training loop.

 1. calculate the loss from a training step
 2. calculate the gradients with the backward method
 3. update the weights based on the gradients
 4. repeat until the predetermined number of epochs is reached

Manually coding this training loop everytime can be inconvenient or a barrier if you’re just getting started with machine learning. Trainer abstracts this process, allowing you to focus on the model, dataset, and training design choices.

The training hyperparameters and options can be easily configured your using `TrainingArguments` which supports many features such as distributed training, torch.compile, mixed precision training, and saving the model to the Hub.

The `TrainingArguments` takes around *120* parameters for configuration!


#### Analyze the custom dataset (ChessMan)

This dataset can be downloaded from https://files.chandrashekar.info/chessman.zip


In [ ]:
##### Step 1: Load your custom dataset

from datasets import load_dataset

dataset = load_dataset("../../data/Chessman-image-dataset/Chess")
dataset

In [ ]:
dataset['train'].features["label"].names

In [ ]:
dataset["train"].features["label"].num_classes

In [ ]:
lbl = dataset["train"].features["label"]
lbl

In [ ]:
lbl.str2int(lbl.names)

In [ ]:
lbl.int2str([3, 2, 1, 1, 0])

In [ ]:
##### Step 2: Split the dataset into training, validation and testing sets

# Split the dataset into training and testing sets
train_test = dataset["train"].train_test_split(test_size=0.2)
train_test


In [ ]:

train_dataset = train_test["train"]
print(train_dataset)

# Further split the testing set into validation and testing sets
test_val = train_test["test"].train_test_split(test_size=0.5)
test_val

In [ ]:
val_dataset = test_val["train"]
test_dataset = test_val["test"]

# Organize the datasets into a dictionary
from datasets import DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})
dataset

In [ ]:
num_classes = dataset["train"].features["label"].num_classes
num_classes

In [ ]:
##### Step 3: Use a pretrained model

checkpoint = "google/vit-base-patch16-224-in21k"

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    checkpoint, num_labels=num_classes
)
print(f"Total parameters: {model.num_parameters()}")
print(f"Trainable parameters: {model.num_parameters(only_trainable=True)}")

In [ ]:
##### Step 4: Preprocess the images suited to the model

from transformers import AutoImageProcessor
image_processor = AutoImageProcessor.from_pretrained(checkpoint, use_fast=True)
image_processor

In [ ]:
img = dataset["train"][0]["image"]
img


In [ ]:
import numpy as np
np.array(img).shape


In [ ]:
i2 = img.convert("RGB")
np.array(i2).shape

In [ ]:

def preprocess_function(examples):
    images = [image.convert("RGB") for image in examples["image"]]
    inputs = image_processor(images=images, return_tensors="pt")
    inputs["label"] = examples["label"]
    return inputs

result = preprocess_function(dataset["train"][:2])
list(result.keys())


In [ ]:

prepared_dataset = dataset.map(preprocess_function, batched=True)
prepared_dataset

In [ ]:
dataset["train"][40]

In [ ]:
prepared_dataset["train"][40]

In [ ]:
##### Step 6: Define the training arguments and Trainer
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../../data/results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    dataloader_pin_memory=False # silence the warning on MacOS Apple Silicon
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["validation"],
)


In [ ]:
##### Step 7: Inference / Predict using the test set
predictions = trainer.predict(prepared_dataset["test"])
predictions.metrics

In [ ]:
predictions.predictions

In [ ]:
##### Step 8: Evaluate the model performance
import numpy as np
predicted_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids
print(true_labels, predicted_labels, sep="\n")


In [ ]:

incorrect_indices = np.where(true_labels != predicted_labels)[0]
incorrect_labels = predicted_labels[incorrect_indices]
true_labels_incorrect = true_labels[incorrect_indices]

print("Incorrectly classified sample indices:", incorrect_indices)
print("Predicted labels for incorrect samples:", incorrect_labels)
print("True labels for incorrect samples:", true_labels_incorrect)


In [ ]:
incorrect_label_names = dataset["test"].features["label"].int2str(incorrect_labels)
true_label_names_incorrect = dataset["test"].features["label"].int2str(true_labels_incorrect)
print("Predicted label names for incorrect samples:", incorrect_label_names)
print("True label names for incorrect samples:", true_label_names_incorrect)

In [ ]:
incorrect_samples = dataset["test"].select(incorrect_indices)
incorrect_samples

In [ ]:
!conda install seaborn -y

In [ ]:
!conda install -c conda-forge scikit-learn -y

In [ ]:
!conda install matplotlib -y

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

predictions = trainer.predict(prepared_dataset["test"])
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
disp.ax_.set_xticklabels(dataset["test"].features["label"].names, rotation=45)
disp.ax_.set_yticklabels(dataset["test"].features["label"].names, rotation=45)
plt.show()

#import seaborn as sns

#cm = confusion_matrix(true_labels, predicted_labels)
#label_names = dataset["train"].features["label"].names

#plt.figure(figsize=(10, 8))
#sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#            xticklabels=label_names, yticklabels=label_names)

#plt.xlabel('Predicted Label')   
#plt.ylabel('True Label')
#plt.title('Confusion Matrix')
#plt.show()

In [ ]:
import sklearn
sklearn.__version__

### Fine-tuning this model

In [ ]:
import evaluate


In [ ]:
##### Define the evaluation metric
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, 
                          references=labels)

In [ ]:
##### Training the model with new dataset (fine-tuning)
import os
#os.environ["TENSORBOAD_LOGGING_DIR"] = "../../data/logs/vit-finetuned-chessman2"

from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="../../data/models/vit-finetuned-chessman2",
    
    num_train_epochs=100,
    
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    weight_decay=0.01,
    
    #logging_dir="../../data/logs/vit-finetuned-chessman2",
    logging_strategy="epoch",

    eval_strategy="epoch", # evaluate every epoch, alternative to "steps"
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    push_to_hub=False,    
    dataloader_pin_memory=False  # silence the warning on MacOS Apple Silicon
)

from transformers import EarlyStoppingCallback
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3,  
    early_stopping_threshold=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]
)
trainer.train()

In [ ]:

trainer.save_model("../../data/models/vit-finetuned-chessman5")

In [ ]:
predictions = trainer.predict(prepared_dataset["test"])
predictions.metrics


In [ ]:
import numpy as np
predicted_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids
print(true_labels, predicted_labels, sep="\n")

In [ ]:
incorrect_indices = np.where(true_labels != predicted_labels)[0]
incorrect_labels = predicted_labels[incorrect_indices]
true_labels_incorrect = true_labels[incorrect_indices]

true_label_names = dataset["test"].features["label"]

print("Incorrectly classified sample indices:", incorrect_indices)
print("Predicted labels for incorrect samples:", incorrect_labels)
print("True labels for incorrect samples:", true_labels_incorrect)

incorrect_label_names = dataset["test"].features["label"].int2str(incorrect_labels)
true_label_names_incorrect = dataset["test"].features["label"].int2str(true_labels_incorrect)
print("Predicted label names for incorrect samples:", incorrect_label_names)
print("True label names for incorrect samples:", true_label_names_incorrect)

In [ ]:
incorrect_samples = dataset["test"].select(incorrect_indices)
incorrect_samples

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
def plot_confusion_matrix(labels, pred_labels):

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = confusion_matrix(labels, pred_labels)
    cm = ConfusionMatrixDisplay(cm, display_labels=range(6))
    cm.plot(values_format='d', cmap='Blues', ax=ax)
    cm.ax_.set_xticklabels(dataset["test"].features["label"].names, rotation=45)
    cm.ax_.set_yticklabels(dataset["test"].features["label"].names, rotation=45)
    plt.show()

plot_confusion_matrix(true_labels, predicted_labels)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(true_labels, predicted_labels)
label_names = dataset["train"].features["label"].names

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
trainer.save_model("../../data/models/vit-finetuned-chessman-final2")

In [ ]:
import matplotlib.pyplot as plt

num_plots = min(5, len(incorrect_samples))

plt.figure(figsize=(10, 5))
for i in range(num_plots):
    idx = incorrect_indices[i]
    # Assuming the dataset returns a dictionary with 'image' and 'label'
    sample = test_dataset[idx] 
    image = sample['image'] 
    
    plt.subplot(1, num_plots, i + 1)
    plt.imshow(image)
    plt.title(f"True: {true_label_names_incorrect[i]}\nPred: {incorrect_label_names[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

---

### Exercise: Try fine-tuning ViT model for fatigue dataset (from Kaggle)
The link to the dataset: https://www.kaggle.com/datasets/rihabkaci99/fatigue-dataset/

Try using `mo-thecreator/vit-Facial-Expression-Recognition` and `trpakov/vit-face-expression` models 
from HuggingFace as the initial checkpoint

In [ ]:
ls ../data/Human_Face_Expression_Fatigue/

In [ ]:
#### Step 1: Load your custom dataset
from datasets import load_dataset
dataset = load_dataset("../data/Human_Face_Expression_Fatigue")
dataset

In [ ]:
#### Step 2: Split the dataset into training, validation and testing sets
train_test = dataset["train"].train_test_split(test_size=0.2)
train_dataset = train_test["train"]
print(train_dataset)

test_val = train_test["test"].train_test_split(test_size=0.5)
val_dataset = test_val["train"]
test_dataset = test_val["test"]

from datasets import DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})
dataset

num_classes = dataset["train"].features["label"].num_classes
num_classes


In [ ]:
#### Step 3: Use a pretrained model
checkpoint = "mo-thecreator/vit-Facial-Expression-Recognition"
model = AutoModelForImageClassification.from_pretrained(
    checkpoint, 
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

In [ ]:
##### Step 4: Preprocess the images suited to the model
from transformers import AutoImageProcessor
image_processor = AutoImageProcessor.from_pretrained(checkpoint, use_fast=True)

def preprocess_function(examples):
    images = [image.convert("RGB") for image in examples["image"]]
    inputs = image_processor(images=images, return_tensors="pt")
    inputs["label"] = examples["label"]
    return inputs

prepared_dataset = dataset.map(preprocess_function, batched=True)
prepared_dataset

In [ ]:
##### Step 5: Setup evaluation metrics and callbacks (optional)
##### Define the evaluation metric
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, 
                          references=labels)

In [ ]:
##### Step 6: Initialize the Trainer and define the training arguments, train the model
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "../data/logs/vit-facial-expression-fatigue"

from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="../data/vit-facial-expression-fatigue",
    
    num_train_epochs=100,
    
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    
    weight_decay=0.01,
    
    #logging_dir="../../data/logs/vit-facial-expression-fatigue",
    logging_strategy="epoch",

    eval_strategy="epoch", # evaluate every epoch, alternative to "steps"
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    push_to_hub=False,    
    dataloader_pin_memory=False  # silence the warning on MacOS Apple Silicon
)

from transformers import EarlyStoppingCallback
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3,  
    early_stopping_threshold=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]
)
trainer.train()

In [ ]:
##### Step 7: Inference / Predict using the test set
predictions = trainer.predict(prepared_dataset["test"])
predictions.metrics


In [ ]:
##### Step 8: Evaluate the model performance with test set and visualize the results

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
predicted_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids
print(true_labels, predicted_labels, sep="\n")

cm = confusion_matrix(true_labels, predicted_labels)
label_names = prepared_dataset["train"].features["label"].names

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
#### Step 9: Save the fine-tuned model
trainer.save_model("../data/models/vit-finetuned-human-face-expression-fatigue")

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ../data/logs/vit-facial-expression-fatigue

---

In [ ]:
g = model.parameters()

In [ ]:
model.get_memory_footprint() / (1024 * 1024)

In [ ]:
total = 0
for p in model.parameters():
    total += p.numel()

total / (1000 * 1000)

---